In [18]:
import duckdb

In [19]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [20]:
df = con.execute("""
                 SELECT * FROM ( 
                    select *, row_number() over(partition by NATBR ORDER BY data_ingestao desc) as row 
                    from bronze_z0019
                    where data_ingestao >= '2025-12-07'
                 ) WHERE row = 1
                 """).fetchdf()
        
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10002,MARTELO,BT50,100,1500,z0019_1.csv,2025-12-07 20:27:01.971089,1
1,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2025-12-07 20:27:01.971089,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2025-12-07 20:29:06.840296,1
3,10003,PREGO,BT10,100,60,z0019_2.csv,2025-12-07 20:29:06.840296,1


In [25]:
df_final = df.drop(columns=['nome_arquivo','data_ingestao','row'])
df_final = df_final.rename(columns={"NATBR":"ID"})
df_final = df_final.rename(columns={"MAKTX":"NM_PRODUTO"})
df_final = df_final.rename(columns={"WERKS":"ID_CATEGORIA"})
df_final = df_final.rename(columns={"MAINS":"ID_FORNECEDOR"})
df_final = df_final.rename(columns={"LABST":"VL_PRECO"})
df_final.head(10)

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10002,MARTELO,BT50,100,1500
1,10001,PARAFUSO,BT10,100,100
2,10004,SERRA,BT50,100,200
3,10003,PREGO,BT10,100,60


In [27]:
df2 = df_final 
df2 = df2.astype({'ID': 'int32',
                   'NM_PRODUTO': 'string',
                   'ID_CATEGORIA': 'string',
                   'ID_FORNECEDOR': 'int32',
                   'VL_PRECO': 'float64'})

df2.head()

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10002,MARTELO,BT50,100,1500.0
1,10001,PARAFUSO,BT10,100,100.0
2,10004,SERRA,BT50,100,200.0
3,10003,PREGO,BT10,100,60.0


In [28]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
    ID bigint,
    NM_PRODUTO TEXT,
    ID_CATEGORIA TEXT,
    ID_FORNECEDOR BIGINT,
    VL_PRECO FLOAT
);
""")

In [30]:
con.execute("""
insert into produtos select * from df2
""")

In [31]:
df_resultado = con.execute(""" select * from produtos """).fetchdf()
df_resultado.head()

,ID,NM_PRODUTO,ID_CATEGORIA,ID_FORNECEDOR,VL_PRECO
0,10002,MARTELO,BT50,100,1500.0
1,10001,PARAFUSO,BT10,100,100.0
2,10004,SERRA,BT50,100,200.0
3,10003,PREGO,BT10,100,60.0


In [32]:
con.close()